## Time Series Grid SANOS Market Generation Test: State 20, Soft, More Strikes

This notebook computes historic time series for implied vols, fitting error, DLV vs a gixed NMS grid.



In [ ]:

import os
os.environ["JAX_ENABLE_X64"] = "True"
import warnings
import numpy as np

warnings.filterwarnings(
    "error",
    message="invalid value encountered in matmul",
    category=RuntimeWarning,
)

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree
%aimport quantdata.widb.options_filter
%aimport quantdata.widb.timeseries
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.sanos.sanos_tools
%aimport quantdata.sanos.arbfree
%aimport quantdata.sanos.sanos
%aimport quantdata.dysanos.grid.interpolated
%aimport quantdata.dysanos.grid.torch_load
%aimport quantdata.dysanos.grid.torch_model
%aimport quantdata.dysanos.grid.torch_gfx 
%aimport quantdata.dysanos.grid.torch_train 


def install_cvxpy_matmul_bound_debug():
    import warnings
    import numpy as np
    import scipy.sparse as sp

    import cvxpy.utilities.bounds as cvx_bounds
    from cvxpy.atoms.affine.binary_operators import MulExpression

    if getattr(cvx_bounds, "_sanos_debug_installed", False):
        return

    cvx_bounds._sanos_debug_installed = True

    orig_matmul_bounds = cvx_bounds.matmul_bounds
    orig_mul_bounds_from_args = MulExpression.bounds_from_args

    def as_array(x):
        if sp.issparse(x):
            return x.toarray()
        return np.asarray(x)

    def describe(name, x):
        x = as_array(x)

        print(f"\n{name}")
        print(f"  shape: {x.shape}")
        print(f"  dtype: {x.dtype}")
        print(f"  nan:   {np.isnan(x).sum() if np.issubdtype(x.dtype, np.number) else 'n/a'}")
        print(f"  +inf:  {np.isposinf(x).sum() if np.issubdtype(x.dtype, np.number) else 'n/a'}")
        print(f"  -inf:  {np.isneginf(x).sum() if np.issubdtype(x.dtype, np.number) else 'n/a'}")
        print(f"  zero:  {(x == 0).sum() if np.issubdtype(x.dtype, np.number) else 'n/a'}")

        finite = np.isfinite(x) if np.issubdtype(x.dtype, np.number) else None
        if finite is not None and finite.any():
            print(f"  finite min/max: {x[finite].min()} / {x[finite].max()}")

    def show_zero_inf_pairs(A, B, label, max_show=20):
        A = as_array(A)
        B = as_array(B)

        if A.ndim == 1:
            A = A.reshape(1, -1)
        if B.ndim == 1:
            B = B.reshape(-1, 1)

        if A.ndim != 2 or B.ndim != 2 or A.shape[1] != B.shape[0]:
            return

        A_zero = A == 0
        B_inf = ~np.isfinite(B)

        count = 0
        print(f"\nPossible 0 * inf terms in {label}:")
        for k in range(A.shape[1]):
            rows = np.flatnonzero(A_zero[:, k])
            cols = np.flatnonzero(B_inf[k, :])
            if len(rows) == 0 or len(cols) == 0:
                continue

            for i in rows[:3]:
                for j in cols[:3]:
                    print(
                        f"  output[{i}, {j}] uses k={k}: "
                        f"A[{i}, {k}]={A[i, k]}, B[{k}, {j}]={B[k, j]}"
                    )
                    count += 1
                    if count >= max_show:
                        return

        if count == 0:
            print("  none found")

    def debug_matmul_bounds(lb1, ub1, lb2, ub2):
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "error",
                    message=r".*invalid value encountered in matmul.*",
                    category=RuntimeWarning,
                )
                return orig_matmul_bounds(lb1, ub1, lb2, ub2)

        except RuntimeWarning as e:
            if "invalid value encountered in matmul" not in str(e):
                raise

            print("\nCVXPY invalid matmul during bounds propagation")
            describe("lb1", lb1)
            describe("ub1", ub1)
            describe("lb2", lb2)
            describe("ub2", ub2)

            # Reconstruct the branch from your traceback:
            # new_lb = a_pos @ lb2 + a_neg @ ub2
            a_pos = np.maximum(as_array(ub1), 0)
            a_neg = np.minimum(as_array(lb1), 0)

            show_zero_inf_pairs(a_pos, lb2, "a_pos @ lb2")
            show_zero_inf_pairs(a_neg, ub2, "a_neg @ ub2")

            raise

    def debug_mul_bounds_from_args(self):
        try:
            return orig_mul_bounds_from_args(self)

        except RuntimeWarning as e:
            if "invalid value encountered in matmul" not in str(e):
                raise

            print("\nCVXPY MulExpression that triggered bad bound matmul")
            print(f"  expression type: {type(self)}")
            print(f"  expression shape: {self.shape}")
            print(f"  expression: {self}")

            for i, arg in enumerate(self.args):
                print(f"\n  arg {i}")
                print(f"    type: {type(arg)}")
                print(f"    shape: {arg.shape}")
                print(f"    is_constant: {arg.is_constant()}")
                print(f"    expr: {arg}")

                if arg.is_constant():
                    describe(f"arg {i}.value", arg.value)

            raise

    cvx_bounds.matmul_bounds = debug_matmul_bounds
    MulExpression.bounds_from_args = debug_mul_bounds_from_args
#install_cvxpy_matmul_bound_debug()

from quantdata.dysanos.grid.torch_load import load, date, Context
r_nms = load("nms", start=date(2020,1,1), grid_num_strikes=40, num_workers=0, verbose=Context.quiet)
print(f"nms /done - found {len(r_nms.ts['dates'])} dates.")
    

/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/dysanos/grid/interpolated.py:201: UserWarning: SPX 2020-01-02: Warning 0.25-smooth SANOS: the maximum grid expiry 1 is greater than the maximum market expiry 0.9651.
/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/dysanos/grid/interpolated.py:201: UserWarning: SPX 2020-01-22: Warning 0.25-smooth SANOS: the maximum grid expiry 1 is greater than the maximum market expiry 0.9135.
/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/dysanos/grid/interpolated.py:201: UserWarning: SPX 2020-01-23: Warning 0.25-smooth SANOS: the maximum grid expiry 1 is greater than the maximum market expiry 0.981.
/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/dysanos/grid/interpolated.py:201: UserWarning: SPX 2020-01-24: Warning 0.25-smooth SANOS: the maximum grid expiry 1 is greater than the maximum market expiry 0.9056.
/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/dysanos/grid/interpolated.py:201: UserWarning: SPX 2020-0

In [ ]:
from cdxcore.bs import bs
from flax.nnx import state

from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.dysanos.grid.torch_model import BaseEnvironment, SanosCompressorConfig, SanosCompressor, FeedForward, torch
from dataclasses import dataclass
import numpy as np
from cdxcore.dynaplot import FigStore, colors_tableau, figure, m_o_m
from quantdata.dysanos.grid.torch_gfx import Display
from quantdata.dysanos.grid.torch_train import train, TrainConfig

mdl_config = SanosCompressorConfig( 
    label="long_big_soft_K40",
    smoothness = 0.25,
    num_state = 20, 
    state_config = FeedForward.Config(
        activation = "SELU",
        num_depth = 2,
        num_width = 200,
        use_bias = True,
    ),
    learn_config = None,
    state_common_init = True
)

trn_config = TrainConfig( num_epochs=100000, lr=1E-3, weight_decay=1e-4, batch_size=128 )
r = train( r_nms, mdl_config=mdl_config, trn_config=trn_config, num_viz_dates=4 )
print("/done")
